In [2]:
import os
os.environ["TANGO_HOST"] = "localhost:9094"   # must be set before importing tango



In [4]:
import tango
scan = tango.DeviceProxy("asyncroscopy/scan/default")
print("State :", scan.State())
print("Status:", scan.status())

State : ON
Status: Idle.


In [6]:
dir(scan)

['Init',
 'State',
 'Status',
 '__attr_cache',
 '__class__',
 '__cmd_cache',
 '__command_inout',
 '__command_inout_asynch_cb',
 '__command_inout_asynch_id',
 '__contains__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get_attr_cache',
 '__get_attr_conf_events',
 '__get_callback_events',
 '__get_cmd_cache',
 '__get_data_events',
 '__get_data_ready_events',
 '__get_devintr_change_events',
 '__get_event_map',
 '__get_event_map_lock',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_orig__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__read_attributes_asynch',
 '__read_attributes_reply',
 '__reduce__',
 '__reduce_ex__',
 '__refresh_attr_cache',
 '__refresh_cmd_cache',
 '__repr__',
 '__setattr__',
 '__setitem__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__subscribe_event_attrib_with_stateless_flag',
 '__subscr

In [11]:
def scan_attributes(device):
    """Every attribute the device advertises, minus the built-in State/Status."""
    return [a for a in device.attribute_list_query() if a.name not in ("State", "Status")]

def read_all(device, show=True):
    """Read every advertised attribute in one Tango call.

    read_attributes does not raise when individual attributes fail — each result
    carries its own has_failed — so one unimplemented _hw_ hook does not hide
    the values that did come back.
    """
    infos = scan_attributes(device)
    names = [a.name for a in infos]
    results = device.read_attributes(names)
    width = max(len(n) for n in names)

    values = {}
    for a, r in zip(infos, results):
        if r.has_failed:
            values[a.name] = None
            if show:
                print(f"  {a.name:<{width}}       FAILED   {r.get_err_stack()[0].desc.splitlines()[0]}")
        else:
            values[a.name] = r.value
            if show:
                unit = "" if a.unit in ("", "No unit") else a.unit
                print(f"  {a.name:<{width}} {r.value!r:>14}   {unit}")
    return values

In [14]:
print("State:", scan.State())
values = read_all(scan)

State: ON
  x_scan_center_m            0.0   m
  y_scan_center_m            0.0   m
  scan_size_m      9.9999997e-06   m
  scan_size_px               256   px
  scan_angle_deg             0.0   deg
  scan_rate_hz        0.49920127   Hz
  probe_x_m             FAILED   TypeError: 'NoneType' object is not subscriptable
  probe_y_m             FAILED   TypeError: 'NoneType' object is not subscriptable


In [13]:
scan.refresh_params()  